In [1]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [7]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients or an image that contains ingredients they have at hand.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [8]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter

# model = ChatOpenRouter(
#     model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
#     temperature=0.8,
# )

model = ChatOpenAI(
    model="google/gemma-4-e4b",
    base_url="http://10.5.0.2:1234/v1")

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [9]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have ground beef, peppers, sage, pita bread, and cheese. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Based on the wonderful ingredients you have—ground beef, peppers, sage, pita bread, and cheese—you are set up to make some incredibly flavorful Mediterranean-inspired meal!

The combination of savory ground beef, aromatic sage, and fresh peppers is perfect for stuffings that warm up beautifully inside crispy pita pockets.

Here are a few suggestions for what we can create:

### 1. Savory Sage & Beef Pita Pockets (Best Bet)
This would be the most direct use of all your ingredients. The ground beef would be seasoned heavily with sage, cooked peppers, and then stuffed into warm pita bread and topped generously with cheese until gooey and melted.

### 2. Mediterranean Stuffed Pita Bake
Instead of just filling the pitas, we could make a slightly more robust meal. We could mix the seasoned ground beef/pepper mixture with some sauce (if you have any tomato sauce on hand) and place it in baking dishes, topped with cheese, and then serve them alongside torn pita bread for dipping.

### 3. Simpl

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have ground beef, peppers, sage, pita bread, and cheese. What can I make?', additional_kwargs={}, response_metadata={}, id='22a55f1c-3f0c-4530-843d-811c1fee723e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 142, 'total_tokens': 235, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 66, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'google/gemma-4-e4b', 'system_fingerprint': 'google/gemma-4-e4b', 'id': 'chatcmpl-idmssx3ez6cgjnli7z6xl', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e523e-5705-7d63-8222-5ab6a1c26f3e-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes with ground beef peppers sage pita bread and cheese'}, 'id': '787237964', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

In [12]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [13]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What can i make with these ingredients?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

config = {"configurable": {"thread_id": "2"}}
response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

What an incredible haul! You are perfectly stocked to make a wide variety of delicious meals with a clear Mediterranean or Italian focus.

Based on your pantry staples—the various pastas (Penne, Spaghetti, Fusilli), rice/quinoa, canned tomatoes and beans, olive oil, and spices—we can easily create several flavorful dishes that require minimal extra shopping.

Here are three suggestions ranging from super quick to a slightly more substantial meal:

### 🍝 1. Mediterranean One-Pot Pasta (The Crowd-Pleaser)
This is a classic comfort dish using the pasta, canned tomatoes, beans, and your favorite spices (like oregano and cumin). It's hearty, customizable, and perfect for feeding a group or making leftovers.

**Key ingredients used:** Pasta, crushed/diced tomatoes, beans (cannellini or chickpeas), olive oil, dried herbs, and onion/garlic powder.

### 🍚 2. Quick Lemon Herb Quinoa Bowl (The Light & Fresh Option)
If you're looking for something lighter, this bowl uses quinoa as the base, mixed 